# Benchmark Dask vs Pandas (4 workers, 6 particiones)

Este notebook crea un dataset grande en 6 archivos, ejecuta el mismo pipeline en Pandas y Dask, y compara tiempos.

Abre el dashboard mientras corres las celdas de Dask: **http://localhost:8787**

## Requisitos previos
1. `docker compose up -d --scale dask-worker=4 dask-scheduler dask-worker`
2. Si el notebook corre fuera de Docker, usa `tcp://localhost:8786`.
3. Si el notebook corre dentro del mismo Docker network, usa `tcp://dask-scheduler:8786`.

In [1]:
import time
from pathlib import Path

import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd
from dask.distributed import Client

pd.set_option('display.max_columns', 50)

In [2]:
# Ajusta esta direccion segun donde corre Jupyter:
SCHEDULER_ADDR = 'tcp://localhost:8786'
# SCHEDULER_ADDR = 'tcp://dask-scheduler:8786'

client = Client(SCHEDULER_ADDR)
client

c:\Users\ser_s\anaconda3\envs\dask\Lib\site-packages\distributed\client.py:1393: VersionMismatchWarning: Mismatched versions found

+-------------+-----------------+-----------------+-----------------+
| Package     | Client          | Scheduler       | Workers         |
+-------------+-----------------+-----------------+-----------------+
| cloudpickle | 3.1.2           | 3.0.0           | 3.0.0           |
| lz4         | None            | 4.3.3           | 4.3.3           |
| msgpack     | 1.1.2           | 1.0.8           | 1.0.8           |
| numpy       | 2.4.4           | 1.26.4          | 1.26.4          |
| pandas      | 3.0.2           | 2.2.2           | 2.2.2           |
| python      | 3.11.15.final.0 | 3.10.12.final.0 | 3.10.12.final.0 |
| toolz       | 1.1.0           | 0.12.0          | 0.12.0          |
| tornado     | 6.5.5           | 6.4             | 6.4             |
+-------------+-----------------+-----------------+-----------------+
  warnings.warn(version_modu

<Client: 'tcp://172.18.0.4:8786' processes=4 threads=56, memory=61.49 GiB>

In [4]:
DATA = Path('benchmark_sales_6parts')
DATA.mkdir(exist_ok=True)

PARTS = 6
ROWS_PER_PART = 800_000  # sube a 1_500_000 o 2_000_000 para exigir mas RAM

def build_part(part_idx: int, rows: int):
    rng = np.random.default_rng(1000 + part_idx)

    start = np.datetime64('2026-01-01')
    end = np.datetime64('2026-04-01')
    seconds = int((end - start) / np.timedelta64(1, 's'))

    order_ts = start + rng.integers(0, seconds, rows).astype('timedelta64[s]')
    region = rng.choice(['Norte', 'Sur', 'Centro', 'Occidente'], size=rows)
    category = rng.choice(['Tecnologia', 'Hogar', 'Deportes', 'Moda'], size=rows)
    customer_id = rng.integers(1, 250_000, size=rows)
    units = rng.integers(1, 10, size=rows)
    unit_price = rng.uniform(10, 700, size=rows).round(2)
    discount = rng.choice([0.0, 0.05, 0.1, 0.15, 0.2], size=rows, p=[0.4, 0.2, 0.2, 0.15, 0.05])
    shipping_cost = rng.uniform(1, 30, size=rows).round(2)

    df = pd.DataFrame({
        'order_id': np.arange(part_idx * rows, (part_idx + 1) * rows),
        'order_ts': order_ts.astype('datetime64[s]'),
        'region': region,
        'category': category,
        'customer_id': customer_id,
        'units': units,
        'unit_price': unit_price,
        'discount': discount,
        'shipping_cost': shipping_cost,
    })

    out = DATA / f'sales_part_{part_idx:02d}.csv'
    df.to_csv(out, index=False)
    return str(out)

# Rebuild dataset
for f in DATA.glob('sales_part_*.csv'):
    f.unlink()

for i in range(PARTS):
    build_part(i, ROWS_PER_PART)

print(f'Dataset creado: {PARTS} archivos x {ROWS_PER_PART:,} filas = {PARTS * ROWS_PER_PART:,} filas')

Dataset creado: 6 archivos x 800,000 filas = 4,800,000 filas


## Pipeline en Pandas (single-process)

In [5]:
t0 = time.perf_counter()

pdf = pd.concat((pd.read_csv(f) for f in sorted(DATA.glob('sales_part_*.csv'))), ignore_index=True)
pdf['order_ts'] = pd.to_datetime(pdf['order_ts'])

pdf['gross_sales'] = pdf['units'] * pdf['unit_price']
pdf['net_sales'] = pdf['gross_sales'] * (1 - pdf['discount'])
pdf['profit'] = pdf['net_sales'] - pdf['shipping_cost'] - (0.55 * pdf['gross_sales'])
pdf['is_high_value'] = pdf['net_sales'] > 1200

pdf = pdf.set_index('order_ts')

p_by_region = pdf.groupby('region')[['net_sales', 'profit']].sum()
p_by_cat_region = pdf.groupby(['region', 'category'])[['net_sales', 'profit']].mean()
p_hourly = pdf['units'].resample('1h').sum()
p_high_value = pdf.groupby('region')['is_high_value'].mean()

pandas_time = time.perf_counter() - t0
pandas_mem_mb = pdf.memory_usage(deep=True).sum() / (1024**2)

print(f'Pandas time: {pandas_time:.2f} s')
print(f'Pandas DataFrame memory: {pandas_mem_mb:.1f} MB')
p_by_region.sort_values('net_sales', ascending=False).head()

Pandas time: 22.43 s
Pandas DataFrame memory: 501.2 MB


,net_sales,profit
region,,
Sur,1.998237e+09,8.072976e+08
Occidente,1.997361e+09,8.070496e+08
Norte,1.997310e+09,8.068948e+08
Centro,1.989812e+09,8.037840e+08


## Pipeline en Dask (distribuido)
Mientras corre esta celda, mira:
- Task Stream
- Workers
- Groups

In [9]:
# Check versions to avoid deserialization error due to environment mismatch
client_version = dask.__version__

def get_dask_version():
    import dask
    return dask.__version__

scheduler_version = client.run_on_scheduler(get_dask_version)
if client_version != scheduler_version:
    raise RuntimeError(f"Version mismatch: client {client_version}, scheduler {scheduler_version}")

t0 = time.perf_counter()

ddf = dd.read_csv(str(DATA / 'sales_part_*.csv'), blocksize=None)
ddf['order_ts'] = dd.to_datetime(ddf['order_ts'])

ddf['gross_sales'] = ddf['units'] * ddf['unit_price']
ddf['net_sales'] = ddf['gross_sales'] * (1 - ddf['discount'])
ddf['profit'] = ddf['net_sales'] - ddf['shipping_cost'] - (0.55 * ddf['gross_sales'])
ddf['is_high_value'] = ddf['net_sales'] > 1200

# Provoca shuffle distribuido visible en dashboard
ddf = ddf.set_index('order_ts').persist()

d_by_region = ddf.groupby('region')[['net_sales', 'profit']].sum()
d_by_cat_region = ddf.groupby(['region', 'category'])[['net_sales', 'profit']].mean()
d_hourly = ddf['units'].resample('1h').sum()
d_high_value = ddf.groupby('region')['is_high_value'].mean()

d_by_region_df, d_by_cat_region_df, d_hourly_s, d_high_value_s = dask.compute(
    d_by_region, d_by_cat_region, d_hourly, d_high_value
)

dask_time = time.perf_counter() - t0

print(f'Dask time: {dask_time:.2f} s')
d_by_region_df.sort_values('net_sales', ascending=False).head()

TypeError: code expected at most 16 arguments, got 18

In [ ]:
comparison = pd.DataFrame({
    'framework': ['pandas', 'dask'],
    'time_s': [pandas_time, dask_time],
})
comparison['speedup_vs_pandas'] = comparison['time_s'].iloc[0] / comparison['time_s']
comparison

In [ ]:
print('Sample outputs (Pandas):')
display(p_by_region.head())
display(p_by_cat_region.head())
display(p_hourly.tail())
display((p_high_value * 100).round(2).astype(str) + '%')

print('Sample outputs (Dask):')
display(d_by_region_df.head())
display(d_by_cat_region_df.head())
display(d_hourly_s.tail())
display((d_high_value_s * 100).round(2).astype(str) + '%')

## Consejos para mostrar mejor la diferencia
- Sube `ROWS_PER_PART` a `1_500_000` o `2_000_000`.
- Mantén 4 workers.
- En dataset pequeño, Pandas puede ser más rápido (overhead de Dask).
- En dataset grande, Dask gana por paralelismo y distribución.